<center><h1>HW3</h1></center>
<br>
<br>

Name: Naveed Alboudwarej 
<br>
USC ID: 8695198359  

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [2]:
import os
import numpy as np
import pandas as pd

Get the AReM Data Set

In [3]:
root= os.path.join("AReM","AReM")
cols= ["time","avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]
frames= []
for activity in os.listdir(root):
    folder= os.path.join(root,activity)
    if not os.path.isdir(folder):
        continue
    for file in os.listdir(folder):
        df= pd.read_csv(os.path.join(folder,file), skiprows= 5, names= cols, usecols= range(7), sep= r"[,\s]+", engine= "python")
        df["activity"]= activity
        df["dataset"]= int(file.replace("dataset","").replace(".csv",""))
        frames.append(df)
data= pd.concat(frames,ignore_index= True)
data.shape

(42239, 9)

### (b) Test and Train Data

In [4]:
test_set= []
train_set= []
for (activity,num),df in data.groupby(["activity","dataset"]):
    if activity.startswith("bending"):
        max_test= 2
    else:
        max_test= 3
    if num<=max_test:
        test_set.append(df)
    else:
        train_set.append(df)
test_set= pd.concat(test_set)
train_set= pd.concat(train_set)

summary= pd.DataFrame({
    "Train":train_set.groupby("activity")["dataset"].nunique(),
    "Test":test_set.groupby("activity")["dataset"].nunique(),
    "All":data.groupby("activity")["dataset"].nunique()})
summary.loc["Total"]= summary.sum()
summary

,Train,Test,All
activity,,,
bending1,5,2,7
bending2,4,2,6
cycling,12,3,15
lying,12,3,15
sitting,12,3,15
standing,12,3,15
walking,12,3,15
Total,69,19,88


### (c) Feature Extraction

#### i. Research

Time-domain features are computed directly from the raw values of a time series, without transforming it into the frequency domain. Commonly used ones for time series classification include:

- **Location:** minimum, maximum, mean, median, and quartiles (typical level and extremes)
- **Spread:** standard deviation, variance, range, interquartile range, mean absolute deviation (how much the signal varies)
- **Shape:** skewness, kurtosis (asymmetry and tail heaviness of the distribution)
- **Energy:** root mean square, absolute energy (overall signal magnitude)
- **Change over time:** mean absolute change, slope/trend, number of peaks, mean-crossing rate (how the signal moves)
- **Dependence:** autocorrelation (how strongly values depend on earlier values)

For this assignment, the features extracted are the minimum, maximum, mean, median, standard deviation, first quartile, and third quartile of each of the 6 time series.

#### ii. Extraction

In [5]:
series= ["avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]
row= data.groupby(["activity","dataset"])
features= pd.DataFrame(index= row.size().index)
for s in series:
    features[s+"_min"]= row[s].min()
    features[s+"_max"]= row[s].max()
    features[s+"_mean"]= row[s].mean()
    features[s+"_median"]= row[s].median()
    features[s+"_std"]= row[s].std()
    features[s+"_q1"]= row[s].quantile(0.25)
    features[s+"_q3"]= row[s].quantile(0.75)
features= features.reset_index()
print(features.shape)
features.head(10)

(88, 44)


,activity,dataset,avg_rss12_min,avg_rss12_max,avg_rss12_mean,avg_rss12_median,avg_rss12_std,avg_rss12_q1,avg_rss12_q3,var_rss12_min,...,avg_rss23_std,avg_rss23_q1,avg_rss23_q3,var_rss23_min,var_rss23_max,var_rss23_mean,var_rss23_median,var_rss23_std,var_rss23_q1,var_rss23_q3
0,bending1,1,37.25,45.00,40.624792,40.50,1.476967,39.2500,42.00,0.0,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.00,1.30
1,bending1,2,38.00,45.67,42.812812,42.50,1.435550,42.0000,43.67,0.0,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.00,1.30
2,bending1,3,35.00,47.40,43.954500,44.33,1.558835,43.0000,45.00,0.0,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
3,bending1,4,33.00,47.75,42.179813,43.50,3.670666,39.1500,45.00,0.0,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
4,bending1,5,33.00,45.75,41.678063,41.75,2.243490,41.3300,42.75,0.0,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
5,bending1,6,37.00,48.00,43.454958,43.25,1.386098,42.5000,45.00,0.0,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
6,bending1,7,36.25,48.00,43.969125,44.50,1.618364,43.3100,44.67,0.0,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83
7,bending2,1,12.75,51.00,24.562958,24.25,3.737514,23.1875,26.50,0.0,...,3.693786,20.5000,27.00,0.0,4.97,0.700187,0.50,0.693720,0.43,0.87
8,bending2,2,0.00,42.75,27.464604,28.00,3.583582,25.5000,30.00,0.0,...,5.053642,15.0000,20.75,0.0,6.76,1.122125,0.83,1.012342,0.47,1.30
9,bending2,3,21.00,50.00,32.586208,33.00,6.238143,26.1875,34.50,0.0,...,5.032424,17.6700,23.50,0.0,13.61,1.162042,0.83,1.332980,0.47,1.30


#### iii. Standard Deviation

In [6]:
x= features.drop(columns= ["activity","dataset"])
np.random.seed(0)
ci= []
for col in x.columns:
    values= x[col].values
    boot_std= []
    for i in range(1000):
        idx= np.random.choice(len(values),len(values),replace= True)
        boot_std.append(values[idx].std(ddof= 1))
    ci.append([col,values.std(ddof= 1),np.percentile(boot_std,5),np.percentile(boot_std,95)])
ci= pd.DataFrame(ci,columns= ["feature","std","lower_5%","upper_95%"]).set_index("feature").round(3)
ci["width_%"]= ((ci["upper_95%"]-ci["lower_5%"])/ci["std"]*100).round(0)
ci

,std,lower_5%,upper_95%,width_%
feature,,,,
avg_rss12_min,9.570,8.266,10.784,26.0
avg_rss12_max,4.394,3.325,5.281,45.0
avg_rss12_mean,5.336,4.706,5.863,22.0
avg_rss12_median,5.440,4.777,6.022,23.0
avg_rss12_std,1.772,1.573,1.954,22.0
avg_rss12_q1,6.154,5.565,6.626,17.0
avg_rss12_q3,5.139,4.325,5.837,29.0
var_rss12_min,0.000,0.000,0.000,NaN
var_rss12_max,5.063,4.626,5.394,15.0


#### iv. Select Features


The three selected features are the **mean**, **maximum** and **standard deviation**. Together they describe the typical level of a series (mean), its peak (maximum) and how much it fluctuates (standard deviation).

The bootstrap table in 1(c)iii supports this choice. For series 1, 2 and 6, the 90% intervals for these features are well above zero, and their widths are mostly 13% to 22% of the estimate (`avg_rss12_max` is wider, at 45%). The features vary across instances and are estimated fairly precisely.

The median, first quartile and third quartile describe the same central tendency as the mean, so they add little. The minimum was avoided because the variance-series minimums are constant or nearly constant: `var_rss12_min` and `var_rss13_min` have a standard deviation of 0 with an interval of [0, 0], and `var_rss23_min` is only 0.046.

## 2. ISLR 3.7.4

### (a) Linear Train

Training RSS, true relationship linear: I'd expect the cubic regression to have a lower or equal training RSS. The cubic model is more flexible, so it can bend to fit the training points more closely, and it can still act like a linear model if the X² and X³ coefficients come out near zero. That means its training RSS can never be higher than the linear model's. Since the data has random noise, the extra flexibility will usually fit some of that noise too, so the cubic RSS ends up slightly lower.


### (b) Linear Test

Test RSS, true relationship linear: I'd expect the linear regression to have the lower test RSS. Since the true relationship is linear, the extra flexibility in the cubic model doesn't help, and it mostly ends up fitting the random noise in the training data. That noise doesn't carry over to new data, so the cubic model does worse on the test set. In other words, it adds variance without reducing any bias.

### (c) Not Linear Train

Training RSS, true relationship not linear: The cubic regression will again have a lower or equal training RSS. Its flexibility lets it follow the curve in the training data more closely than a straight line can, and it can always fall back to the linear fit, so it can't do worse on the training set.

### (d) Not Linear Testing

Test RSS, true relationship not linear: There isn't enough information to tell which model will have the lower test RSS. It depends on how far the true relationship is from linear. If it's only a little curved, the cubic model's extra flexibility mostly adds variance, and the linear model could win. If it's strongly non-linear, the cubic model's lower bias should give it the lower test RSS. This is the bias-variance trade-off.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 